# Notebook 3 — Privacy with Anonymizer

**Hands-on synthetic data pipeline workshop with NeMo Data Designer and NeMo Anonymizer — PyData London 2026**

## What you'll build

By the end of this notebook you'll have an anonymized version of production-style VLM usage logs. You'll inspect where identifiers enter the trace, preview Anonymizer's detections, compare that coverage with a regex baseline, then run `Substitute` over the full log column.

## The setup

Imagine the rich-business-document VLM from Notebook 2 is now deployed. Real users are uploading internal documents — clinical trial status reports, financial variance memos, product launch readiness plans, operations dashboard exports — and the team wants to study **how the model is being used**: what tasks people attempt, where the model struggles, and what kinds of documents arrive in the wild.

Those logs are useful, but they are not neutral text. The system prompt may include the authenticated user's name, email, employer, role, and project codename. Uploaded documents may include stakeholder names, document IDs, budgets, KPIs, and dates. Tool results can add even more identifiers that the user never typed directly.

That is the problem [NeMo Anonymizer](https://nvidia-nemo.github.io/Anonymizer/latest/) handles: detect identifying entities, replace them using a strategy you choose, and produce a version of the same dataset that more people can inspect safely.

## What you'll learn

1. **PII detection as a pipeline stage** — GLiNER NER plus an LLM augmenter/validator, both running on hosted endpoints.
2. **Four Replace-mode strategies** — `Substitute`, `Redact`, `Annotate`, `Hash` — and the privacy/utility/audience tradeoffs between them.
3. **`preview()` before committing** — the Rich highlight view shows you exactly what the detector caught (and what it didn't) before you spend tokens on the full dataset.
4. **`data_summary` matters** — the cheapest quality lever Anonymizer gives you. Always set it.
5. **Replace vs. Rewrite** — when per-entity replacement is enough vs. when you need the full text rewritten to suppress *inferable* identifiers (we'll Replace today and only mention Rewrite at the end).

## How this fits

The shape is the same as the earlier notebooks: define the pipeline, preview a small sample, inspect the result, then scale up. Here the pipeline stage is privacy rather than generation or judging.

## Setup

Note the package vs. import naming: dependency is `nemo-anonymizer`, but you import from `anonymizer`. `uv sync` already installed it for you.

In [ ]:
import json
from collections import Counter

import pandas as pd

from notebook_helpers import (
    ARTIFACT_DIR,
    DATA_DIR,
    build_anonymizer_model_setup,
    environment_setup,
    load_usage_logs_seed,
)

# Pick which hosted backend to use. Leave as "auto" to use whichever key is in
# your .env (precedence: NVIDIA -> OpenRouter -> OpenAI), or set explicitly to
# "nvidia", "openrouter", or "openai" to flip between providers without editing
# any other files.
PROVIDER = "openrouter"

provider = environment_setup(provider=PROVIDER)

## Part 1 — Load the deployment logs

We use `data/usage_logs_seed.parquet` — VLM deployment logs in production ChatML shape (one session per row, same schema you would export from an API logging pipeline). Each row is one **multi-turn user session**. Instructors can regenerate the file with [`bonus_generate_usage_logs.ipynb`](bonus_generate_usage_logs.ipynb); the checked-in copy is what we anonymize here. The columns:

| Column | Description |
|---|---|
| `user_persona`, `task_intent`, `document_type` | sampler metadata |
| `system_prompt` | application-built system prompt naming the authenticated user, employer, role, project codename — shown for inspection; the canonical log copy lives at `messages[0]` of `conversation_trace` |
| `attached_document_context` | text representation of the rich business document the user uploaded (clinical trial report, financial variance memo, market research brief, product launch readiness plan, operations dashboard export, customer support incident review) — also shown for inspection |
| `conversation_trace` | full multi-turn ChatML trace as a JSON string — **system message at index 0**, then user messages, assistant tool calls, tool results, assistant replies. This is the canonical log shape (matches OpenAI / Nemotron API log format) and the column we'll anonymize. |

### Where does PII enter your logs? Four surfaces — all inside `conversation_trace`.

Real deployment logs leak identifying information through *four* independent surfaces, and they all live inside the trace because that's how production-quality logs are stored:

| Surface | Where in the trace | What an attacker / casual reader sees |
|---|---|---|
| System prompt | `messages[0]` (`role: "system"`) | the user's name, email, employer, project codename — *every single log line* |
| User messages | `messages[*]` with `role: "user"` | task references, occasional colleague names or codenames |
| Tool calls | `messages[*].tool_calls[*].custom.input` | JSON arguments quote organization + document IDs |
| Tool results | `messages[*]` with `role: "tool"` | *additional* PII the user never typed (CRM contacts, owner emails, contract IDs, ARR bands, reviewer names) |

The system prompt and tool results are easy to overlook. The system prompt is attached to every conversation, and tool results can introduce identifiers the user never typed. Since all four surfaces live inside `conversation_trace`, this notebook anonymizes that single canonical log column. The standalone `system_prompt` and `attached_document_context` columns are kept for inspection but are not separately anonymized; downstream consumers should read from the anonymized `conversation_trace` instead.

In [ ]:
logs = load_usage_logs_seed()
print(f"Loaded {len(logs)} log rows.\n")
logs[["user_persona", "task_intent", "document_type"]].head(5)

In [ ]:
# Show one full row of text content so you can see the leaks before we touch anything.
row = logs.iloc[0]
print("═══ system_prompt (application-built) ═══")
print(row["system_prompt"])
print("\n═══ attached_document_context (uploaded doc) ═══")
print(row["attached_document_context"])
print("\n═══ conversation_trace (multi-turn ChatML, pretty-printed) ═══")
trace = json.loads(row["conversation_trace"])
for i, msg in enumerate(trace["messages"]):
    role = msg.get("role")
    if role == "tool":
        print(f"[{i}] tool         (id={msg.get('tool_call_id')})  {msg.get('content','')[:100]}…")
    elif msg.get("tool_calls"):
        for tc in msg["tool_calls"]:
            cust = tc.get("custom") or {}
            print(f"[{i}] assistant 🛠  {cust.get('name')}({cust.get('input')})")
    else:
        print(f"[{i}] {role:<12} {(msg.get('content') or '')[:100]}...")

🔍 **Look at what's in each layer.**

- The **system prompt** alone names the user, their employer, their email, their role, their project codename, and 1–2 colleagues. Every log line carries this string.
- The **attached document** names the organization plus 2–4 stakeholders (often with emails), a document ID, and 3–6 figures (enrollment counts, dollar variances, KPI values, severity levels, etc.).
- The **conversation trace** layers more on top: tool-call argument JSON quotes organization + document IDs; tool results return CRM contacts, owner emails, and reviewer names the user *never typed*; assistant replies echo all of it back.

No single field looks especially dramatic. The risk comes from accumulation: names, roles, document IDs, account context, tool results, and assistant replies all reinforce each other across many sessions. A reviewer reading thousands of traces is no longer just debugging model behavior; they are also reading a detailed operational record of the users and teams behind those traces.

## Part 2 — Configure Anonymizer

Anonymizer is a small, focused tool built on top of Data Designer. Its pipeline:

1. **Basic NER detection** — a zero-shot NER model, GLiNER, scans for entity spans (names, emails, phone numbers, organizations, addresses, amounts, …). It runs on NVIDIA Build when Build is selected; OpenRouter/OpenAI runs use the workshop Brev GLiNER endpoint. No local weights required.
2. **LLM augmenter / validator** — a text LLM extends coverage on entities GLiNER missed and validates spans.
3. **Replacement** — applies your chosen strategy to each detected span. `Substitute` calls an LLM per entity to generate consistent substitutions; `Redact` / `Annotate` / `Hash` are local string transforms.

### Configuration

The Anonymizer constructor takes:

- `model_providers` — one or two `ModelProvider` entries (your API backend, plus the workshop Brev GLiNER host when you are not on Build)
- `model_configs` — a YAML string defining the model pool plus `selected_models:` blocks naming which alias plays each detector / validator / replacer role

`build_anonymizer_model_setup()` in `notebook_helpers.py` picks models **per provider** — not the same mapping as Data Designer:

| Provider | GLiNER | LLM stages (validate / augment / replace) |
|---|---|---|
| **NVIDIA Build** | `nvidia/gliner-pii` on Build | `openai/gpt-oss-120b` on Build |
| **OpenRouter** | workshop Brev endpoint | `openai/gpt-oss-120b` on OpenRouter (same model as Build) |
| **OpenAI** | workshop Brev endpoint | `gpt-5.4` (no nemotron or gpt-oss on api.openai.com) |

OpenRouter attendees still use `qwen/qwen3.6-flash` in Notebooks 1–2; only this notebook swaps the chat model. Constants and alternates (e.g. `qwen/qwen3.7-max`) live in `notebook_helpers.py`.

In [ ]:
from anonymizer import (
    Anonymizer,
    AnonymizerConfig,
    AnonymizerInput,
    Substitute,
)

model_providers, model_configs_yaml = build_anonymizer_model_setup(provider)
print("Anonymizer model_configs YAML:")
print(model_configs_yaml)

anonymizer = Anonymizer(
    model_providers=model_providers,
    model_configs=model_configs_yaml,
)

Anonymizer reads input from a file path or URL, **one text column at a time**. It only touches the column you name — extra columns in the parquet are ignored. We point it straight at the checked-in seed file.

**Important:** always set `data_summary`. It is the single cheapest quality lever Anonymizer gives you — both detection and replacement use it as context, so the augmenter knows *what kind of text it's looking at*. A one-line description is enough.

In [ ]:
SEED_PATH = (DATA_DIR / "usage_logs_seed.parquet").resolve()

DATA_SUMMARY = (
    "VLM deployment logs for a document-QA copilot: one ChatML JSON trace per row "
    "(system prompt with authenticated user, user turns, tool calls, tool results, replies). "
    "Business PII throughout prose and embedded JSON — names, emails, organizations, "
    "roles, document IDs, KPIs, and internal project codenames."
)

TEXT_COLUMN = "conversation_trace"
anonymizer_input = AnonymizerInput(
    source=str(SEED_PATH),
    text_column=TEXT_COLUMN,
    data_summary=DATA_SUMMARY,
)
print(
    f"Anonymizer input: {len(logs)} rows from {SEED_PATH.relative_to(DATA_DIR.parent)}, "
    f"column {TEXT_COLUMN!r}"
)

## Part 3 — Preview with `Substitute`

`Substitute` is the production default for analytics-style log workflows: for each detected entity it asks an LLM to generate a *contextually plausible* replacement. **"Wescott Logistics" might become "Halberg Freight"**. Sentence flow stays natural — the user message still reads like a user message; assistant replies still read like model output. That naturalness preserves analytical signal: you can still read the logs and see *what kind of question the user was asking* and *what the model did with each tool call*.

It costs one LLM call per detected entity, which is why we always start with a `preview()` to inspect detector output before paying for the full run.

Because `preview()` runs on a small sample, it is the right place to inspect detections before committing to the full run.

In [ ]:
from anonymizer import configure_logging, LoggingConfig
configure_logging(LoggingConfig.debug())

In [ ]:
substitute_config = AnonymizerConfig(replace=Substitute())
substitute_preview = anonymizer.preview(
    config=substitute_config, data=anonymizer_input, num_records=3,
)
substitute_preview.display_record()

🔍 **Inspect what the detector caught.** The Rich-highlighted display shows each detected entity with its label and replacement. Look critically:

- Did organization names and named stakeholders get tagged as `organization` / `person`?
- Were document IDs, monetary variances, and KPI figures recognized?
- Are there **false positives** — something tagged as PII that isn't actually identifying (e.g. a generic chart axis label)?
- Are there **false negatives** — something obviously identifying that slipped through (e.g. an internal project codename, a colleague's first name)?

PII detection is never perfect. The point of `preview()` is that you see this *before* you commit to the full run. If you're seeing systematic gaps, there are knobs: lower `gliner_threshold` for recall (default 0.3, try 0.2), or extend the entity-label list with domain terms via `Detect(entity_labels=[*DEFAULT_ENTITY_LABELS, ...])`.

The next cell hand-codes regex patterns for the entity types you'd reach for first with basic tooling — emails, phone numbers, dates, ZIP codes, SSNs, IP addresses, credit cards — and counts them on the **same preview rows**. The markdown after that cell explains how to read the comparison against Anonymizer.

In [ ]:
import re

from IPython.display import display

# Basic-tools baseline: closed-set patterns you'd ship without an extra dependency.
BASELINE_PATTERNS = {
    "EMAIL_ADDRESS": re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),
    "PHONE_NUMBER": re.compile(r"\+?\d[\d\s().-]{7,}\d"),
    "DATE_TIME": re.compile(r"\b\d{4}-\d{2}-\d{2}\b"),
    "US_ZIP_CODE": re.compile(r"\b\d{5}(?:-\d{4})?\b"),
    "US_SSN": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "IP_ADDRESS": re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
    "CREDIT_CARD": re.compile(r"\b(?:\d[ -]?){13,16}\b"),
}

# Apples-to-apples: same N rows the Substitute preview ran on.
n_preview = len(substitute_preview.trace_dataframe)
texts = logs.head(n_preview)[TEXT_COLUMN].astype(str).tolist()

baseline_counts = {
    label: sum(len(pat.findall(t)) for t in texts)
    for label, pat in BASELINE_PATTERNS.items()
}
baseline_total = sum(baseline_counts.values())

regex_df = pd.DataFrame({"regex_matches": baseline_counts})
regex_df.loc["TOTAL"] = baseline_total

# `final_entities` is shaped {"entities": [{label, value, ...}, ...]} in
# Anonymizer 0.2.0; older versions exposed a plain list — we handle both.
def _row_entities(row):
    if row is None:
        return []
    if isinstance(row, list):
        return row
    if isinstance(row, dict):
        ent = row.get("entities")
        return list(ent) if ent is not None else []
    ent = getattr(row, "entities", None)
    return list(ent) if ent is not None else []

trace_df = substitute_preview.trace_dataframe
if "final_entities" in trace_df.columns:
    all_entities = [e for r in trace_df["final_entities"] for e in _row_entities(r)]
    label_breakdown = Counter(
        (ent.get("label") if isinstance(ent, dict) else getattr(ent, "label", "?")) or "?"
        for ent in all_entities
    )
    n_anonymizer = len(all_entities)
    anon_df = pd.DataFrame(
        label_breakdown.most_common(12),
        columns=["label", "anonymizer_entities"],
    ).set_index("label")
else:
    n_anonymizer = 0
    anon_df = pd.DataFrame(columns=["anonymizer_entities"])

display(regex_df)
display(anon_df)

pd.DataFrame(
    [{
        "preview_rows": n_preview,
        "regex_total": baseline_total,
        "anonymizer_total": n_anonymizer,
        "gap": n_anonymizer - baseline_total,
    }]
)

### Regex vs Anonymizer — reading the comparison

Both tables above use the **same preview rows** you just inspected.

| | Regex baseline | Anonymizer preview |
|---|---|---|
| **What it catches** | Closed-set patterns you ship yourself (email, phone, ISO date, …) | GLiNER spans + LLM validation + LLM augmentation |
| **Typical scale** | A few dozen matches on structured tokens | Often an order of magnitude more — names, orgs, document IDs, occupations, codenames |

The **gap** column is the practical answer to *"why not just regex this away?"* Regex never ships labels for internal project codenames, invented product names, compound role descriptors, or organization names buried in JSON tool arguments — yet those dominate deployment logs like ours.

Closed-set regex still belongs in the toolkit (cheap pre-filter, compliance checklists). It is not a substitute for domain-aware detection on ChatML traces.

## Part 4 — A note on `Rewrite` mode (we won't run it here)

Everything above is **Replace mode** -- Anonymizer detects entity spans and replaces them in place. That's the right tool for **structured data where the structure has to survive anonymization**: log traces, JSON tool calls, ChatML messages, OCR'd document fields. After anonymization, the JSON still has to parse and the message roles still have to align.

Anonymizer also has a **Rewrite mode** that uses an LLM to rephrase an entire passage against a `PrivacyGoal`. It's the right tool for **free-form text where content matters and structure is flexible** -- narrative customer-support transcripts, free-text reviews, incident-postmortem write-ups, candidate-feedback notes. In those domains a sentence-level rewrite can scrub *inferable* identifiers (writing style, an unusually specific combination of facts) that no span detector can put a box around.

Rewrite mode is **not the right call for this notebook's data**. A sentence-level rewrite would break the JSON structure of `conversation_trace`; after anonymization, the trace still needs to parse as ChatML-style messages with roles, tool calls, and tool results intact. It is still worth knowing when Rewrite mode is the right tool.


## Part 5 — Full run and save

For this notebook we'll use `Substitute`, which is usually the best fit when downstream consumers need to read or analyze the logs. For a narrower audit workflow, `Redact`, `Annotate`, or `Hash` may be a better fit.

In [ ]:
# `anonymizer.run()` processes the full input parquet (all 20 rows). On the
# free tier this takes ~4-5 minutes -- a comfortable narration window for
# walking through the four PII surfaces, the regex-baseline gap above, and
# what `Substitute` mode is actually doing under the hood.
result = anonymizer.run(config=substitute_config, data=anonymizer_input)
n_failed = len(result.failed_records or [])
print(f"✓ Anonymized {TEXT_COLUMN}: {len(result.dataframe)} ok, {n_failed} failed")

**If you saw any non-zero `failed`** — those are infra issues (rate limits, auth, transient network errors), not a strategy problem. Inspect `results[col].failed_records` to see the per-record reason. Fix that *before* tweaking strategy.

In [ ]:
# Row order matches the seed parquet — join back to `logs` by index if you need metadata.
anonymized = pd.DataFrame(
    {f"{TEXT_COLUMN}_anonymized": result.dataframe[f"{TEXT_COLUMN}_replaced"].tolist()}
)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
out_path = ARTIFACT_DIR / "03_usage_logs_anonymized.parquet"
anonymized.to_parquet(out_path, index=False)
print(f"Saved {len(anonymized)} anonymized traces to {out_path.relative_to(ARTIFACT_DIR.parent)}")
anonymized.head(3)

## Recap

You now have a complete dataset lifecycle:

**sample → seed → generate → judge → filter → anonymize**

Privacy is now an explicit pipeline stage. The same shape that produced training data in Notebooks 1 and 2 also produced the deployment logs (`bonus_generate_usage_logs.ipynb`), and Anonymizer applies the same declarative idea to privacy: configure, preview, run.

## Where to go next

- **Production-grade VLM SDG:** the [VLM long-document recipes](https://github.com/NVIDIA-NeMo/DataDesigner/tree/main/docs/assets/recipes/vlm_long_doc) on GitHub are the full version of Notebook 2 — eight stages including OCR, classification-first filtering, multi-page windowed QA, whole-document QA, and a frontier judge. Background story is in the [iterative SDG dev note](https://nvidia-nemo.github.io/DataDesigner/latest/devnotes/training-a-vlm-to-understand-long-documents-an-iterative-sdg-story/).
- **More Data Designer recipes:** [text-to-SQL, code generation, agent search trajectories, deep research](https://github.com/NVIDIA-NeMo/DataDesigner/tree/main/docs/assets/recipes).
- **Anonymizer Rewrite mode:** [introducing-nemo-anonymizer](https://nvidia-nemo.github.io/Anonymizer/latest/devnotes/introducing-nemo-anonymizer-text-anonymization-for-the-reasoning-era/) — the dev note covering Rewrite mode, `PrivacyGoal`, and inferable-identifier suppression.

## One principle to take with you

Treat dataset creation — *and the data your model produces in deployment* — as engineering. Distributions, dependencies, validation, filtering, privacy: these are pipeline stages, not afterthoughts. Define the pipeline, inspect intermediate outputs, document the tradeoffs, and only then scale up.